# 2장 1강: 역전파와 기울기 문제 — 실습문제 풀이

> **풀이 기준**: 프로젝트에 업로드된 `concrete_data.csv`로 코드를 실제 실행한 결과입니다. 출력은 소수점 여섯째 자리에서 반올림했고, 계산에는 반올림 전 값을 사용했습니다.
>
> **가정한 내용**: 별도 판단이 필요한 부분은 없었습니다. 문제에 주어진 초기값·식·학습률을 그대로 사용했습니다.

---

## 실습 준비

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("concrete_data.csv")
df.columns = df.columns.str.strip()
print("전체 데이터 크기:", df.shape)
print(df[["cement", "concrete_compressive_strength"]].head(3).to_string(index=False))

전체 데이터 크기: (1030, 9)
 cement  concrete_compressive_strength
  540.0                          79.99
  540.0                          61.89
  332.5                          40.27


---

## 실습 1. 역전파로 기울기를 구하고 파라미터 수정하기

### 문제 1-1. 첫 샘플의 순전파·손실·기울기 계산

In [2]:
x = df["cement"].to_numpy()[0] / 100.0
y = df["concrete_compressive_strength"].to_numpy()[0]

w1 = 0.5
b1 = 0.1
w2 = 20.0
b2 = 20.0

# 순전파
z1 = w1 * x + b1
h1 = np.maximum(0, z1)
y_hat = w2 * h1 + b2

# 오차와 손실
error = y_hat - y
loss = 0.5 * error ** 2

# 역전파 (z1 > 0이므로 relu_slope = 1)
print("z1 > 0 인가?:", z1 > 0)
relu_slope = 1.0
grad_y_hat = y_hat - y
grad_w2 = grad_y_hat * h1
grad_b2 = grad_y_hat
grad_h1 = grad_y_hat * w2
grad_z1 = grad_h1 * relu_slope
grad_w1 = grad_z1 * x
grad_b1 = grad_z1

print("x =", x, " y =", y)
print("z1 =", round(z1, 6), " h1 =", round(h1, 6), " y_hat =", round(y_hat, 6))
print("error =", round(error, 6), " loss =", round(loss, 6))
print("grad_y_hat =", round(grad_y_hat, 6), " grad_h1 =", round(grad_h1, 6), " grad_z1 =", round(grad_z1, 6))
print("grad_w1 =", round(grad_w1, 6), " grad_b1 =", round(grad_b1, 6))
print("grad_w2 =", round(grad_w2, 6), " grad_b2 =", round(grad_b2, 6))

z1 > 0 인가?: True
x = 5.4  y = 79.99
z1 = 2.8  h1 = 2.8  y_hat = 76.0
error = -3.99  loss = 7.96005
grad_y_hat = -3.99  grad_h1 = -79.8  grad_z1 = -79.8
grad_w1 = -430.92  grad_b1 = -79.8
grad_w2 = -11.172  grad_b2 = -3.99


#### 답변

**수식 및 계산 과정**

- 순전파: z1 = 0.5 × 5.4 + 0.1 = **2.8** → h1 = ReLU(2.8) = **2.8** → ŷ = 20 × 2.8 + 20 = **76**
- 오차: error = 76 − 79.99 = **−3.99**
- 손실: L = ½ × (−3.99)² = ½ × 15.9201 = **7.96005**
- 역전파
  - grad_y_hat = −3.99
  - grad_w2 = −3.99 × 2.8 = **−11.172**, grad_b2 = **−3.99**
  - grad_h1 = −3.99 × 20 = **−79.8**
  - grad_z1 = −79.8 × 1 = **−79.8**
  - grad_w1 = −79.8 × 5.4 = **−430.92**, grad_b1 = **−79.8**

**Q1. z1, h1, y_hat, error, loss는 각각 얼마인가요? 음수 error는 무엇을 뜻하나요?**

- z1 = 2.8, h1 = 2.8, ŷ = 76, error = −3.99, loss = 7.96005입니다.
- error가 음수라는 것은 모델 출력(76)이 실제 정답(79.99)보다 **3.99만큼 작다**는 뜻입니다.

**Q2. 출력층의 grad_w2와 grad_b2는 각각 얼마인가요? 왜 h1을 곱하는 것은 가중치 쪽인가요?**

- grad_w2 = −11.172, grad_b2 = −3.99입니다.
- ŷ = w2 × h1 + b2에서 w2는 h1과 곱해져 있으므로, w2를 조금 바꾸면 ŷ는 h1배만큼 변합니다. 반면 b2는 그냥 더해지므로 ŷ에 1배로 영향을 줍니다. 그래서 가중치 기울기에만 h1이 곱해집니다.

**Q3. grad_h1, grad_z1, grad_w1, grad_b1은 각각 얼마인가요? 왜 수정 전 w2를 사용하나요?**

- grad_h1 = −79.8, grad_z1 = −79.8, grad_w1 = −430.92, grad_b1 = −79.8입니다.
- 기울기는 **현재 상태(수정 전 파라미터)에서 손실이 어떻게 변하는지**를 나타냅니다. 모든 기울기를 같은 시점의 값으로 구해야 일관된 방향을 얻을 수 있으므로, 출력층을 먼저 바꾼 뒤 그 값으로 은닉층 기울기를 구하면 안 됩니다.

**Q4. 모든 파라미터 기울기가 음수라는 것은 파라미터 값도 음수라는 뜻인가요? 기울기와 실제 정답의 역할은 어떻게 다른가요?**

- 아닙니다. 파라미터 값(0.5, 0.1, 20, 20)은 모두 양수입니다. 기울기가 음수라는 것은 **그 파라미터를 조금 늘리면 손실이 줄어드는 방향**이라는 뜻입니다.
- 실제 정답은 손실을 계산할 때 출력과 비교하는 기준값이고, 기울기는 그 손실을 줄이려면 각 파라미터를 어느 방향으로 얼마나 움직일지 알려 주는 값입니다.

**Q5. 역전파로 기울기를 구한 시점에 파라미터 수정도 끝난 것인가요?**

- 아닙니다. 역전파는 기울기를 **계산**하는 단계일 뿐이며, 이 시점에 w1·b1·w2·b2는 그대로입니다. 수정은 다음 단계(문제 1-2)에서 학습률을 적용해 별도로 합니다.

---

### 문제 1-2. 기울기로 네 파라미터를 한 번 수정하기

In [3]:
lr = 0.0001

w1_new = w1 - lr * grad_w1
b1_new = b1 - lr * grad_b1
w2_new = w2 - lr * grad_w2
b2_new = b2 - lr * grad_b2

z1_new = w1_new * x + b1_new
h1_new = np.maximum(0, z1_new)
y_hat_new = w2_new * h1_new + b2_new
loss_new = 0.5 * (y_hat_new - y) ** 2

print("w1:", w1, "→", round(w1_new, 6))
print("b1:", b1, "→", round(b1_new, 6))
print("w2:", w2, "→", round(w2_new, 6))
print("b2:", b2, "→", round(b2_new, 6))
print("출력:", round(y_hat, 6), "→", round(y_hat_new, 6))
print("손실:", round(loss, 6), "→", round(loss_new, 6))
print("손실이 줄었나?:", loss_new < loss)

w1: 0.5 → 0.543092
b1: 0.1 → 0.10798
w2: 20.0 → 20.001117
b2: 20.0 → 20.000399
출력: 76.0 → 80.817332
손실: 7.96005 → 0.342239
손실이 줄었나?: True


#### 답변

**수식 및 계산 과정 (새 파라미터 = 기존 − lr × 기울기)**

- w1_new = 0.5 − 0.0001 × (−430.92) = 0.5 + 0.043092 = **0.543092**
- b1_new = 0.1 − 0.0001 × (−79.8) = 0.1 + 0.00798 = **0.10798**
- w2_new = 20 − 0.0001 × (−11.172) = 20 + 0.0011172 = **20.001117**
- b2_new = 20 − 0.0001 × (−3.99) = 20 + 0.000399 = **20.000399**
- 다시 순전파: z1 = 0.543092 × 5.4 + 0.10798 ≈ 3.040677 → ŷ ≈ 20.001117 × 3.040677 + 20.000399 ≈ **80.817332**
- 새 손실: ½ × (80.817332 − 79.99)² ≈ **0.342239**

**Q1. 네 파라미터의 새 값은 각각 얼마인가요? 음수 기울기를 뺐는데 왜 값이 커지나요?**

- w1 0.543092, b1 0.10798, w2 20.001117, b2 20.000399입니다.
- 음수를 빼는 것은 그 크기만큼 더하는 것과 같습니다. 예를 들어 0.5 − (0.0001 × (−430.92)) = 0.5 + 0.043092입니다.

**Q2. 수정 전후 출력과 손실은 각각 얼마인가요? 수정 후 출력이 실제 정답보다 커졌는데도 손실이 줄었나요?**

- 출력 76 → 80.817332, 손실 7.96005 → 0.342239입니다.
- 출력이 정답(79.99)을 0.827332만큼 넘었지만, 정답과의 거리는 3.99에서 약 0.83으로 줄었으므로 손실도 줄었습니다. 손실은 차이를 제곱하므로 부호와 관계없이 거리가 가까울수록 작습니다.

**Q3. 새 출력 계산에서 은닉값 h1도 다시 구해야 하는 이유는 무엇인가요?**

- h1은 w1·b1으로 계산되는 값인데, w1·b1이 바뀌었으므로 h1도 2.8에서 약 3.040677로 바뀝니다. 이전 h1을 재사용하면 은닉층 수정의 효과가 빠진 잘못된 출력이 됩니다.

**Q4. 첫 샘플의 손실이 한 번 줄었다는 결과로 전체 데이터에서 좋은 모델이라고 결론 내릴 수 있나요?**

- 없습니다. 샘플 하나에 맞춰 한 번 수정했을 뿐이며, 다른 샘플(예: 과제의 세 번째 샘플)에서는 오히려 다른 방향의 수정이 필요할 수 있습니다. 전체 성능은 여러 샘플에서의 손실을 보고 판단해야 합니다.

---

## 실습 2. 기울기 소실과 폭주를 숫자로 확인하기

### 문제 2-1. 역으로 전달되는 기울기의 반복 곱 비교

In [4]:
steps = np.array([1, 3, 5, 10])
g_start = 1.0

g_small = g_start * (0.1 ** steps)
g_same = g_start * (1.0 ** steps)
g_large = g_start * (2.0 ** steps)

print("steps  :", steps)
print("g_small:", g_small)
print("g_same :", g_same)
print("g_large:", g_large)

steps  : [ 1  3  5 10]
g_small: [1.e-01 1.e-03 1.e-05 1.e-10]
g_same : [1. 1. 1. 1.]
g_large: [   2.    8.   32. 1024.]


#### 답변

**수식 및 계산 과정**

- 계수 0.1을 세 번: 1 × 0.1 × 0.1 × 0.1 = **0.001**
- 계수 2.0을 세 번: 1 × 2 × 2 × 2 = **8**

**Q1. 세 단계를 거친 작은 계수·기준 계수·큰 계수의 결과는 각각 얼마인가요?**

- 0.1 → **0.001**, 1.0 → **1**, 2.0 → **8**입니다.

**Q2. 열 단계를 거친 결과는 각각 얼마이며, 어느 경우가 소실·폭주를 설명하나요?**

- 0.1 → **1e-10 (0.0000000001)**, 1.0 → **1**, 2.0 → **1024**입니다.
- 계수 0.1은 기울기가 거의 0이 되므로 **기울기 소실**, 계수 2.0은 기울기가 매우 커지므로 **기울기 폭주**를 설명합니다.

**Q3. 앞쪽 파라미터에 전달되는 기울기가 너무 작거나 너무 크면 파라미터 수정에 어떤 문제가 생길 수 있나요?**

- 너무 작으면 lr × 기울기가 거의 0이 되어 입력 쪽 층의 파라미터가 거의 바뀌지 않으므로 학습이 매우 느려지거나 사실상 멈춥니다.
- 너무 크면 한 번에 파라미터가 크게 바뀌어 손실이 오히려 커지거나 값이 튀어 학습이 불안정해집니다(심하면 NaN 발생).

**Q4. 기울기가 작으면 언제나 소실, 크면 언제나 폭주라고 판단할 수 있나요? 이 예제에서 무엇을 고정해 비교했나요?**

- 아닙니다. 최적점 근처에서는 기울기가 자연히 작고, 오차가 큰 초기에는 기울기가 클 수 있습니다. 소실·폭주는 **여러 층을 거치며 반복 곱으로 크기가 체계적으로 줄거나 커지는 현상**을 가리킵니다.
- 이 예제에서는 출발 기울기(1)와 단계 수를 고정하고, 매 단계의 전달 계수만 0.1·1.0·2.0으로 바꿔 비교했습니다.

**Q5. ReLU가 양수에서 미분값 1을 갖는다는 사실만으로 소실·폭주가 모두 해결된다고 할 수 있나요?**

- 없습니다. 전달 계수에는 활성화 함수의 미분뿐 아니라 **가중치**도 곱해집니다(실습 1의 grad_h1 = grad_y_hat × w2). 가중치가 크면 폭주, 작으면 소실이 생길 수 있고, 음수 구간에서는 ReLU 미분이 0이라 기울기가 끊길 수도 있습니다.

---

## 과제. 같은 역전파와 파라미터 수정 복습

### 문제 3-1. 세 번째 샘플로 한 번의 학습 계산 수행

In [5]:
task_x = df["cement"].to_numpy()[2] / 100.0
task_y = df["concrete_compressive_strength"].to_numpy()[2]
task_w1 = 0.5
task_b1 = 0.1
task_w2 = 20.0
task_b2 = 20.0
task_lr = 0.0001

# 순전파
task_z1 = task_w1 * task_x + task_b1
task_h1 = np.maximum(0, task_z1)
task_y_hat = task_w2 * task_h1 + task_b2
task_error = task_y_hat - task_y
task_loss = 0.5 * task_error ** 2

# 역전파
task_relu_slope = 1.0  # task_z1 > 0
task_grad_y_hat = task_error
task_grad_w2 = task_grad_y_hat * task_h1
task_grad_b2 = task_grad_y_hat
task_grad_h1 = task_grad_y_hat * task_w2
task_grad_z1 = task_grad_h1 * task_relu_slope
task_grad_w1 = task_grad_z1 * task_x
task_grad_b1 = task_grad_z1

# 수정
task_w1_new = task_w1 - task_lr * task_grad_w1
task_b1_new = task_b1 - task_lr * task_grad_b1
task_w2_new = task_w2 - task_lr * task_grad_w2
task_b2_new = task_b2 - task_lr * task_grad_b2

# 다시 순전파
task_z1_new = task_w1_new * task_x + task_b1_new
task_h1_new = np.maximum(0, task_z1_new)
task_y_hat_new = task_w2_new * task_h1_new + task_b2_new
task_loss_new = 0.5 * (task_y_hat_new - task_y) ** 2

print("x =", task_x, " y =", task_y)
print("z1 =", round(task_z1, 6), " h1 =", round(task_h1, 6), " y_hat =", round(task_y_hat, 6))
print("error =", round(task_error, 6), " loss =", round(task_loss, 6))
print("grad_w1 =", round(task_grad_w1, 6), " grad_b1 =", round(task_grad_b1, 6),
      " grad_w2 =", round(task_grad_w2, 6), " grad_b2 =", round(task_grad_b2, 6))
print("새 파라미터: w1 =", round(task_w1_new, 6), " b1 =", round(task_b1_new, 6),
      " w2 =", round(task_w2_new, 6), " b2 =", round(task_b2_new, 6))
print("출력:", round(task_y_hat, 6), "→", round(task_y_hat_new, 6))
print("손실:", round(task_loss, 6), "→", round(task_loss_new, 6), " 감소?:", task_loss_new < task_loss)

# 별도 복습: 기울기 전달
print("1.0 * (0.1 ** 3) =", 1.0 * (0.1 ** 3))
print("1.0 * (2.0 ** 3) =", 1.0 * (2.0 ** 3))

x = 3.325  y = 40.27
z1 = 1.7625  h1 = 1.7625  y_hat = 55.25
error = 14.98  loss = 112.2002
grad_w1 = 996.17  grad_b1 = 299.6  grad_w2 = 26.40225  grad_b2 = 14.98
새 파라미터: w1 = 0.400383  b1 = 0.07004  w2 = 19.99736  b2 = 19.998502
출력: 55.25 → 48.021072
손실: 112.2002 → 30.039556  감소?: True
1.0 * (0.1 ** 3) = 0.0010000000000000002
1.0 * (2.0 ** 3) = 8.0


#### 답변

**수식 및 계산 과정**

- 순전파: z1 = 0.5 × 3.325 + 0.1 = **1.7625**, h1 = **1.7625**, ŷ = 20 × 1.7625 + 20 = **55.25**
- 오차·손실: error = 55.25 − 40.27 = **14.98**, L = ½ × 14.98² = ½ × 224.4004 = **112.2002**
- 역전파: grad_w2 = 14.98 × 1.7625 = **26.40225**, grad_b2 = **14.98**, grad_h1 = grad_z1 = 14.98 × 20 = **299.6**, grad_w1 = 299.6 × 3.325 = **996.17**, grad_b1 = **299.6**
- 수정: w1 = 0.5 − 0.099617 = **0.400383**, b1 = 0.1 − 0.02996 = **0.07004**, w2 = 20 − 0.002640225 ≈ **19.99736**, b2 = 20 − 0.001498 = **19.998502**
- 다시 계산: z1 ≈ 1.401313, ŷ ≈ **48.021072**, 손실 ≈ **30.039556**

**Q1. 수정 전 z1, h1, 출력, 오차, 손실은 각각 얼마인가요?**

- z1 = 1.7625, h1 = 1.7625, ŷ = 55.25, error = 14.98, loss = 112.2002입니다.

**Q2. 네 파라미터의 기울기는 각각 얼마인가요? 실습의 첫 샘플과 부호가 다른 이유는 무엇인가요?**

- grad_w1 = 996.17, grad_b1 = 299.6, grad_w2 = 26.40225, grad_b2 = 14.98로 모두 양수입니다.
- 첫 샘플은 출력(76)이 정답(79.99)보다 **작아** error가 음수였지만, 세 번째 샘플은 출력(55.25)이 정답(40.27)보다 **커서** error가 양수입니다. 모든 기울기가 error에서 출발하므로(h1, w2, x는 양수) 부호가 함께 바뀝니다.

**Q3. 네 파라미터를 한 번 수정한 값과 수정 후 출력·손실은 얼마인가요? 파라미터가 증가했나요, 감소했나요?**

- w1 0.400383, b1 0.07004, w2 19.99736, b2 19.998502이며, 출력 55.25 → 48.021072, 손실 112.2002 → 30.039556입니다.
- 기울기가 양수이므로 네 파라미터 모두 **감소**했고, 그 결과 너무 컸던 출력이 정답 쪽으로 줄었습니다.

**Q4. 기울기 1에 계수 0.1과 2를 세 번 곱한 결과는 각각 얼마이며, 어떤 현상을 설명하나요?**

- 0.1³ = **0.001** → 기울기가 점점 작아지는 **기울기 소실**
- 2³ = **8** → 기울기가 점점 커지는 **기울기 폭주**

**Q5. 이번 과제가 첫 샘플의 학습을 이어서 수행한 것인가요? 손실 감소로 어디까지 확인할 수 있나요?**

- 아닙니다. 실습의 `_new` 값을 쓰지 않고 원래 초기값(0.5, 0.1, 20, 20)에서 다시 시작한 **독립적인 계산**입니다.
- 손실 감소로 확인할 수 있는 것은 "이 샘플에서 기울기 반대 방향으로 한 번 움직였더니 이 샘플의 손실이 줄었다"까지입니다. 전체 데이터에서의 성능이나 여러 번 반복한 학습 결과는 알 수 없습니다.

---

## 실습 마무리

**1. 순전파와 역전파의 진행 방향과 역할은 어떻게 다른가요?**

- 순전파는 입력 → 은닉층 → 출력 방향으로 진행하며 출력과 손실을 계산합니다.
- 역전파는 손실 → 출력층 → 은닉층 방향으로 거슬러 올라가며 각 파라미터의 기울기를 계산합니다.

**2. 역전파와 파라미터 수정은 같은 작업인가요?**

- 다릅니다. 역전파는 기울기를 **구하는** 단계이고, 파라미터 수정은 그 기울기와 학습률로 새 값을 **만드는** 단계입니다(새 값 = 기존 − lr × 기울기).

**3. 기울기가 음수일 때 기존 값에서 lr×기울기를 빼면 어떻게 되나요?**

- 음수를 빼므로 파라미터가 **커집니다**. 실습 1에서 w1이 0.5에서 0.543092로 커진 것이 그 예입니다.

**4. 기울기 소실과 폭주를 각각 설명하세요.**

- 소실: 역전파에서 1보다 작은 값이 반복 곱해져, 입력 쪽 층에 도달하는 기울기가 거의 0이 되어 학습이 거의 일어나지 않는 현상입니다.
- 폭주: 1보다 큰 값이 반복 곱해져 기울기가 지나치게 커지고, 파라미터가 크게 튀어 학습이 불안정해지는 현상입니다.

**5. 같은 출력 기울기 1에서 0.1과 2를 반복해서 곱하는 이유는 무엇인가요?**

- 출발점을 같게 고정해야 전달 계수의 크기만이 결과를 좌우한다는 것을 비교할 수 있기 때문입니다. 이를 통해 층이 깊어질수록 계수가 1보다 작으면 소실, 크면 폭주가 일어난다는 원리를 확인했습니다.

**6. 이번 실습으로 전체 콘크리트 데이터의 모델 성능을 확인했나요?**

- 아닙니다. 시멘트량 한 특성과 샘플 두 개(첫 번째, 세 번째)에서 각각 한 번씩 계산했을 뿐입니다. 전체 데이터에 대한 학습과 성능 평가는 수행하지 않았습니다.